In [1]:
from tqdm.auto import tqdm
import os
import re
import unicodedata
import torch
import pandas as pd

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftConfig, PeftModel

# -----------------------------
# Offline mode
# -----------------------------
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

# -----------------------------
# Paths
# -----------------------------
test_path = "/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv"
adapter_path = "/kaggle/input/models/jiayuanshe/hunyuan-ada/transformers/default/1"
base_model_path = "/kaggle/input/models/jiayuanshe/hunyuan-7b-base/transformers/default/1"

# -----------------------------
# Normalization
# Must match training exactly
# -----------------------------
def normalize_transliteration(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()

    replacements = {
        "š": "sh",
        "sz": "sh",
        "ṣ": "s.",
        "s,": "s.",
        "ṭ": "t.",
        "t,": "t.",
        "ḫ": "h.",
        "h,": "h.",
        "ĝ": "g.",
        "g,": "g.",
        "ā": "a",
        "ē": "e",
        "ī": "i",
        "ū": "u",
        "é": "e",
        "í": "i",
        "ú": "u",
        "á": "a",
        "â": "a",
        "à": "a",
        "è": "e",
        "ì": "i",
        "ù": "u",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    text = re.sub(r"\[\.\.\.\]", "...", text)
    text = re.sub(r"\[|\]", "", text)
    text = re.sub(r"\(|\)", "", text)
    text = re.sub(r"\{.*?\}", "", text)
    text = re.sub(r"\^", "", text)
    text = re.sub(r"<{2,}", "<", text)
    text = re.sub(r">{2,}", ">", text)

    text = unicodedata.normalize("NFKC", text)

    text = text.replace(".", "-")
    text = text.replace("-", "")

    text = re.sub(r"\s+", " ", text)
    text = text.strip()

    return text

# -----------------------------
# Output cleanup
# -----------------------------
def clean_translation_output(text):
    if not isinstance(text, str):
        return ""

    text = text.strip()

    # Remove common chat artifacts if they appear
    text = re.sub(r"^\s*(assistant|translation)\s*:\s*", "", text, flags=re.I)
    text = text.split("\n")[0].strip()

    # Remove wrapping quotes if the model adds them
    if len(text) >= 2 and text[0] == text[-1] and text[0] in ['"', "'"]:
        text = text[1:-1].strip()

    return text

# -----------------------------
# Load test data
# -----------------------------
test_ds = pd.read_csv(test_path)
test_ds["transliteration"] = test_ds["transliteration"].apply(normalize_transliteration)

# -----------------------------
# Load model + tokenizer
# -----------------------------
print("Checking files...")
print("Adapter path exists:", os.path.exists(adapter_path))
print("Base model path exists:", os.path.exists(base_model_path))
print("Adapter files:", os.listdir(adapter_path))
print("Base model files:", os.listdir(base_model_path))

print("\nLoading PEFT config...")
peft_config = PeftConfig.from_pretrained(
    adapter_path,
    local_files_only=True,
)
print("Adapter expects base model:", peft_config.base_model_name_or_path)

print("\nLoading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    device_map="auto",
    local_files_only=True,
    trust_remote_code=True,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)

# Prefer base tokenizer unless you know the adapter tokenizer differs intentionally
print("Loading tokenizer from adapter path...")
tokenizer = AutoTokenizer.from_pretrained(
    adapter_path,
    local_files_only=True,
    trust_remote_code=True,
)

print("Attaching adapter...")
model = PeftModel.from_pretrained(
    base_model,
    adapter_path,
    local_files_only=True,
)

# Optional: merge adapter for faster inference
try:
    model = model.merge_and_unload()
    print("Merged LoRA adapter into base model.")
except Exception as e:
    print("Could not merge adapter, continuing without merge:", repr(e))

model.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.eos_token_id

if tokenizer.bos_token_id is not None:
    model.config.bos_token_id = tokenizer.bos_token_id
    model.generation_config.bos_token_id = tokenizer.bos_token_id

print("Model loaded successfully!")

# -----------------------------
# Correct device for device_map="auto"
# -----------------------------
def get_input_device(model):
    try:
        return model.get_input_embeddings().weight.device
    except Exception:
        return next(model.parameters()).device

input_device = get_input_device(model)
print("Using input device:", input_device)

# -----------------------------
# Translation function
# Keep prompt format and normalization unchanged
# -----------------------------
def translate_akkadian(transliteration, model=model, tokenizer=tokenizer):
    transliteration = normalize_transliteration(transliteration)

    messages = [
        {
            "role": "user",
            "content": (
                "Translate the following segment into English, without additional explanation.\n\n"
                f"{transliteration}"
            ),
        },
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )

    inputs = {k: v.to(input_device) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,
            num_beams=50,
            early_stopping=True,
            repetition_penalty=1.02,
            no_repeat_ngram_size=3,
        )

    prompt_len = inputs["input_ids"].shape[1]
    decoded = tokenizer.decode(
        outputs[0][prompt_len:],
        skip_special_tokens=True,
    )

    return clean_translation_output(decoded)

# -----------------------------
# Sanity check
# -----------------------------
print("\nSanity check on first 3 samples...")
for i in range(min(3, len(test_ds))):
    src = str(test_ds.iloc[i]["transliteration"])
    pred = translate_akkadian(src)
    print(f"\nSample {i}")
    print("SRC :", src[:200])
    print("PRED:", pred[:200])

# -----------------------------
# Run inference
# -----------------------------
print("\nRunning inference on test dataset...")
translations = []
test_transliterations = test_ds["transliteration"].fillna("").tolist()

print(f"Processing {len(test_transliterations)} samples...")

for i, translit in enumerate(tqdm(test_transliterations)):
    try:
        translations.append(translate_akkadian(translit))
    except Exception as e:
        print(f"\nError on sample index {i}")
        print(f"Text preview: {str(translit)[:150]}...")
        print(f"Exception type: {type(e).__name__}")
        print(f"Exception repr: {repr(e)}")
        translations.append("")

test_ds["translation"] = translations

# -----------------------------
# Save results
# -----------------------------
output_path = "submission.csv"
test_ds = test_ds[["id", "translation"]]
test_ds.to_csv(output_path, index=False)

print(f"✓ Saved predictions to {output_path}")
print("\nSample predictions:")
print(test_ds.head())

Checking files...
Adapter path exists: True
Base model path exists: True
Adapter files: ['adapter_model.safetensors', 'training_metrics.csv', 'adapter_config.json', 'README.md', 'tokenizer.json', 'tokenizer_config.json', 'training_config.json', 'chat_template.jinja', 'special_tokens_map.json']
Base model files: ['model.safetensors.index.json', 'model-0002-of-0004.safetensors', 'config.json', 'README.md', 'tokenizer.json', 'tokenizer_config.json', 'model-0003-of-0004.safetensors', 'License.txt', 'model-0004-of-0004.safetensors', 'model-0001-of-0004.safetensors', 'generation_config.json']

Loading PEFT config...


`torch_dtype` is deprecated! Use `dtype` instead!
Unrecognized keys in `rope_parameters` for 'rope_type'='dynamic': {'beta_fast', 'rope_theta', 'beta_slow', 'mscale_all_dim', 'mscale', 'alpha'}


Adapter expects base model: ./hunyuan

Loading base model...


Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading tokenizer from adapter path...
Attaching adapter...
Merged LoRA adapter into base model.
Model loaded successfully!
Using input device: cuda:0

Sanity check on first 3 samples...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Sample 0
SRC : umma karuum kaniiama ana aaqiil datim aiiprini kaar kaarma u wabarratim qibi„ma muppuum aa alimki ilikam
PRED: Thus says the Kanesh colony: speak to the datu-payers, to our messengers, to the colony and to the traders. A letter from the City has arrived.

Sample 1
SRC : ina muppiim aa alimki iatu u„miim anim mamaan kuan iaaumuni ina nemilim daaur ula ewa iaratiau karuum kaniia ilaqe
PRED: According to the tablet of the City, no one has claimed any meteoric iron from me since this day. In the favorable decision of Assur, the karum of Kanesh will take my iron.

Sample 2
SRC : kima muppini taaameani amakam lu ana aimiim ana egallim idiin lu teraat egallim ukalim lu naaima adini la idiin mala kuan naaau nibi„it aaiim auumau u aumi abi„au ina muppiim luuptanimma iati aiiprini
PRED: As soon as you have heard our letter, either give it to a merchant or to the palace, or keep it as a document of the palace or sell it. Until now he has not sold it. Write in the letter how much o

  0%|          | 0/4 [00:00<?, ?it/s]

✓ Saved predictions to submission.csv

Sample predictions:
   id                                        translation
0   0  Thus says the Kanesh colony: speak to the datu...
1   1  According to the tablet of the City, no one ha...
2   2  As soon as you have heard our letter, either g...
3   3  Send a copy of our letter to every single karu...


In [2]:
! nvidia-smi

Wed Mar 11 18:24:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P0             37W /   70W |   10095MiB /  15360MiB |     47%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----